# `fusit.dataset` prototype

*(paired with `trace_prototype.ipynb`, which exercises the cue tagger)*

Exercises every part of [`fusit/dataset.py`](../src/fusit/dataset.py) against the corpora in
[`dataset/`](../dataset) and checks the loaders against the raw jsonl, so a wrong filter or a
silently dropped record shows up here rather than in a sweep.

Run top to bottom. Every assertion should pass silently; anything printed is there to be read.

**Kernel**: `Python (dpfusion-repro)` — the env where `fusit` is installed editable. It is preselected in this notebook's metadata; if your Jupyter does not see it, run `python -m ipykernel install --user --name dpfusion-repro` from that env.

## 1. Where the data is

In [ ]:
from pathlib import Path

from fusit.dataset import DATASET_DIR, DATASETS, Dataset, Item, SynthPAI, Synthetic, SyntheticItem
from fusit.dataset import ATTACK_ENTITY_TYPES, Span, TAB_ENTITY_TYPES, TabDocument, TabECHR
from fusit.dataset import get_dataset, load_dataset

print(f"DATASET_DIR : {DATASET_DIR}")
print(f"DATASETS    : {DATASETS}")
print()
for name in DATASETS:
    p = get_dataset(name).path
    rel = p.relative_to(DATASET_DIR) if DATASET_DIR in p.parents else p.name
    # stat() only after exists(): tab_echr is gitignored, so a fresh clone will not have it
    size = f"{p.stat().st_size/1e6:7.2f} MB" if p.exists() else "  not fetched"
    print(f"{name:10s} {str(rel):28s} {'OK' if p.exists() else 'MISSING':8s} {size}")

## 2. Loading

`get_dataset` memoizes the instance and the instance memoizes the parse, so the jsonl is read
once per process no matter how many subsamples you ask for. The timing below shows the cold
parse against the warm one.

In [ ]:
import time

fresh = SynthPAI()                      # bypass the memo to time a cold parse
t0 = time.perf_counter(); fresh.items(); cold = time.perf_counter() - t0
t0 = time.perf_counter(); fresh.items(); warm = time.perf_counter() - t0
print(f"cold parse : {cold*1000:7.1f} ms")
print(f"warm  call : {warm*1000:7.3f} ms  ({cold/max(warm,1e-9):,.0f}x faster)")

assert get_dataset("synthpai") is get_dataset("synthpai"), "get_dataset should memoize"

synthpai, synthetic = get_dataset("synthpai"), get_dataset("synthetic")
print()
print(f"{synthpai!r}  -> {len(synthpai.items())} items")
print(f"{synthetic!r} -> {len(synthetic.items())} items")

## 3. What an item looks like

Both datasets share `.username` / `.text` / `.relevant_pii`, which is what lets callers stay
dataset-agnostic. `SyntheticItem` adds `.hardness`.

In [ ]:
def show(item, chars=260):
    print(f"type        : {type(item).__name__}")
    print(f"username    : {item.username}")
    print(f"relevant_pii: {item.relevant_pii}")
    extra = {k: v for k, v in vars(item).items() if k not in {"username", "text", "relevant_pii"}}
    if extra:
        print(f"extra       : {extra}")
    print(f"text ({len(item.text)} chars):")
    print("  " + item.text[:chars].replace("\n", "\n  ") + ("..." if len(item.text) > chars else ""))

show(synthpai.select(n=1, seed=3)[0])
print("\n" + "-" * 78 + "\n")
show(synthetic.select(n=1, seed=3)[0])

### The contract, checked over every item

The cell above eyeballs one item from each corpus. This one sweeps all 823 and asserts the
promise the rest of the codebase leans on — that `.username` / `.text` / `.relevant_pii` are
always there and always usable, so `repro/` can take either dataset from
`get_dataset(name).select(...)` without branching on which one it got.

Uniqueness of `username` is the assertion that earns its keep. Sweep results are keyed by
`(dataset, item, row)` and merged across shard files, so two items sharing an id do not raise
anything — one silently overwrites the other. TRACE-RPS's own `"<age><sex>"` id for the
synthetic corpus is not unique, which is why `Synthetic.load` indexes by record instead.

The last two asserts pin down where the corpora legitimately *differ*: synthetic items are
`SyntheticItem` (so `.hardness` exists) and carry exactly one target attribute, where a
SynthPAI profile carries 4.2 on average.

Assertions print nothing when they hold, so a single line of output means all of it passed.

In [ ]:
# the shared contract, checked explicitly
for ds in (synthpai, synthetic):
    for it in ds.items():
        assert isinstance(it, Item)
        assert it.username and isinstance(it.username, str)
        assert it.text.strip()
        assert it.relevant_pii and all(isinstance(v, str) and v for v in it.relevant_pii.values())
    ids = [i.username for i in ds.items()]
    assert len(ids) == len(set(ids)), f"{ds.name}: duplicate usernames"

assert all(isinstance(i, SyntheticItem) for i in synthetic.items())
assert all(len(i.relevant_pii) == 1 for i in synthetic.items()), "synthetic items target exactly one attribute"
print("contract holds for both datasets (unique ids, non-empty text, non-empty ground truth)")

## 4. Corpus statistics

In [ ]:
def table(rows, headers):
    widths = [max(len(str(r[i])) for r in [headers] + rows) for i in range(len(headers))]
    line = "  ".join(h.ljust(w) for h, w in zip(headers, widths))
    print(line); print("-" * len(line))
    for r in rows:
        print("  ".join(str(c).ljust(w) for c, w in zip(r, widths)))

for name in DATASETS:
    print(f"=== {name} ===")
    for k, v in get_dataset(name).stats().items():
        print(f"  {k:20s} {round(v, 2) if isinstance(v, float) else v}")
    print()

In [ ]:
# attribute coverage side by side -- the two corpora use the SAME attribute vocabulary,
# which is what makes cross-dataset comparison meaningful
sp, sy = synthpai.stats()["attribute_counts"], synthetic.stats()["attribute_counts"]
attrs = sorted(set(sp) | set(sy))
table([[a, sp.get(a, 0), sy.get(a, 0)] for a in attrs], ["attribute", "synthpai", "synthetic"])

assert set(sp) == set(sy) == set(SynthPAI.ATTRIBUTES), "attribute vocabularies should match"
print("\nboth corpora cover all 8 SynthPAI attributes")

### What `hardness` means

A 1-5 label, annotated by hand and carried on the raw record (`Synthetic.load` reads it
straight through). It rates **how obliquely the text gives the attribute away** — not text
length, not how well any model does on it.

Two records from the corpus, both with ground truth `Toronto, Canada`:

> **hardness 1** — "Here in **Toronto** (and by extension all of **Canada**), we're quite famous
> for Poutine..."
>
> **hardness 5** — "We have the enchanting gorges over at **Rouge**, where I love to walk,
> especially when the leaves are changing..."

The first says the city outright. The second never names a place: you have to know Rouge is a
national urban park in Toronto and read the fall-foliage detail as a climate hint.

That is the axis this whole project turns on. An NER-based defence removes `Toronto` and
handles the first case; it cannot see the second at all, because there is no entity to tag.
Splitting results by hardness is how the explicit-PII and implicit-PII regimes get told apart
— it is why `repro/cue_tagger.py` adds attention- and CoT-derived signals on top of NER.

Two caveats worth holding onto:

- The label is **per attribute**, so it only survives on a corpus where each item has exactly
  one target. SynthPAI profiles carry many attributes, and there hardness is consumed by the
  `hardness >= 1 and certainty >= 1` filter rather than kept on the item — which is why this
  histogram exists for `synthetic` only.
- It is a human judgement about how explicit the cue is, **not** measured model difficulty.
  The records also carry a `guess_correctness` field, and the two disagree often: a model
  still gets many hardness-5 items right, and misses a few hardness-1 ones.

In [ ]:
# synthetic carries a per-item difficulty; SynthPAI's hardness is folded into the filter
hc = synthetic.stats()["hardness_counts"]
total = sum(hc.values())
for h in sorted(hc):
    bar = "#" * round(40 * hc[h] / max(hc.values()))
    print(f"hardness {h}  {hc[h]:4d}  {100*hc[h]/total:5.1f}%  {bar}")

# ...and which attributes are hardest to infer, per the same labels
import json

raw = [json.loads(l) for l in open(synthetic.path)]
per_feature = {}
for r in raw:
    per_feature.setdefault(r["feature"], []).append(r["hardness"])
print()
table([[f, len(hs), round(sum(hs) / len(hs), 2)]
       for f, hs in sorted(per_feature.items(), key=lambda kv: -sum(kv[1]) / len(kv[1]))],
      ["attribute", "n", "mean hardness"])

## 5. Sampling is reproducible

`select` is the only entry point the sweeps use, so its determinism is what makes an
interrupted run resumable.

In [ ]:
a = [i.username for i in synthpai.select(n=10, seed=0)]
b = [i.username for i in synthpai.select(n=10, seed=0)]
c = [i.username for i in synthpai.select(n=10, seed=1)]
assert a == b, "same seed must give the same sample, in the same order"
assert a != c, "different seeds should differ"
print("seed 0:", a[:4], "...")
print("seed 1:", c[:4], "...")

# Does a larger n extend the smaller sample, or redraw it? Observed: with the same seed it
# extends -- select(20) starts with exactly the select(10) items. That falls out of how
# CPython's random.sample works and is NOT a documented guarantee, so treat it as an
# observation, not something to build on. sweep_cue.py does not rely on it: it gets its
# priority-first ordering by calling select twice and de-duplicating.
big = [i.username for i in synthpai.select(n=20, seed=0)]
print(f"\nselect(20, seed=0) starts with the select(10, seed=0) items: {big[:10] == a}")

In [ ]:
# n=None (or n >= corpus size) returns everything, as a fresh list each time
allp = synthpai.select()
assert len(allp) == len(synthpai.items())
assert synthpai.select(n=10**6) == allp
assert allp is not synthpai.items(), "select must not hand out the cached list"

allp.reverse()                                   # mutating the result must not disturb the cache
assert synthpai.select()[0].username != allp[0].username or len(allp) == 1
print("select() returns a fresh list; the cache is safe from caller mutation")

# load_dataset is the one-shot form
assert [i.username for i in load_dataset("synthpai", n=5, seed=0)] == \
       [i.username for i in synthpai.select(n=5, seed=0)]
print("load_dataset(name, ...) == get_dataset(name).select(...)")

## 6. Cross-check against the raw jsonl

The loaders apply filters copied from TRACE-RPS. Re-derive the expected counts straight from
the files here, so a filter that silently drops records gets caught.

In [ ]:
import json

raw_sp = [json.loads(l) for l in open(synthpai.path) if l.strip()]
print(f"raw synthpai records      : {len(raw_sp)}")
print(f"loaded profiles           : {len(synthpai.items())}")

# the filter is hardness >= 1 AND certainty >= 1 on reviews.human
def expected_pii(rec):
    h = rec["reviews"].get("human", {})
    return {a: e["estimate"] for a in SynthPAI.ATTRIBUTES
            if isinstance(e := h.get(a), dict)
            and e.get("hardness", 0) >= 1 and e.get("certainty", 0) >= 1 and e.get("estimate")}

kept = [r for r in raw_sp if expected_pii(r)]
print(f"records passing the filter: {len(kept)}")
assert len(kept) == len(synthpai.items()), "loader and re-derived filter disagree"

by_name = {i.username: i for i in synthpai.items()}
for r in kept:
    assert by_name[r["username"]].relevant_pii == expected_pii(r), r["username"]
print("\nevery profile's ground truth matches the re-derived filter")
print(f"profiles dropped by the filter: {len(raw_sp) - len(kept)}")

In [ ]:
# text is the comments joined with newlines, same as TRACE's run_reddit_anonymization
r = raw_sp[0]
assert by_name[r["username"]].text == "\n".join(c["text"] for c in r["comments"])
print(f"{r['username']}: {len(r['comments'])} comments -> {len(by_name[r['username']].text)} chars, join verified")

In [ ]:
raw_sy = [json.loads(l) for l in open(synthetic.path) if l.strip()]
print(f"raw synthetic records : {len(raw_sy)}")
print(f"loaded items          : {len(synthetic.items())}")

dropped_income = [r for r in raw_sy if r["feature"] == "income"]
dropped_no_gt  = [r for r in raw_sy if r["feature"] != "income" and not r["personality"].get(r["feature"])]
print(f'  dropped, feature == "income" : {len(dropped_income)}')
print(f"  dropped, no ground truth     : {len(dropped_no_gt)}")
assert len(raw_sy) - len(dropped_income) - len(dropped_no_gt) == len(synthetic.items())

# ground truth and hardness come straight from the record
first = synthetic.items()[0]
idx = int(first.username[3:7])
assert first.relevant_pii == {raw_sy[idx]["feature"]: str(raw_sy[idx]["personality"][raw_sy[idx]["feature"]])}
assert first.hardness == int(raw_sy[idx]["hardness"])
print(f"\n{first.username} traces back to raw record {idx}: ground truth and hardness match")

## 7. TAB-ECHR — a different shape of ground truth

The two corpora above ask *which attribute value can be inferred*. TAB-ECHR asks *which
characters are private*, and that difference goes all the way down: a `TabDocument` carries
typed `Span`s instead of `relevant_pii`, and it is deliberately **not** an `Item`. Giving it
an empty `relevant_pii` to force it into the same contract would put a lie on every document
and quietly break anything that trusts that field.

This is DP-Fusion's own evaluation corpus (paper Section 5.1): hand-annotated European Court
of Human Rights judgments, with private information marked in eight categories.

Two annotation choices have to be made when reading the file, and `TabECHR` makes both
explicit rather than burying them:

- **Identifier types.** TAB marks each mention `DIRECT`, `QUASI` or `NO_MASK`. Appendix A.5:
  "we do not distinguish between direct and quasi identifiers. Instead, we take their union
  and treat all such values uniformly." `NO_MASK` is dropped.
- **Annotator.** Documents carry 1-10 annotators. `"first"` is arbitrary but reproducible and
  matches this repo's earlier runs; `"union"` takes every annotator's spans. Appendix A.17
  argues recall is the axis that matters — a missed span falls outside the DP guarantee
  entirely, while an over-tagged one only costs utility.

Unlike the other two, the splits are **not vendored**: 70 MB of json, gitignored. Fetch them
from [our Drive copy](https://drive.google.com/file/d/14oTnnNoSlJy6z9__BfpzTebYTu2-ePwT/view)
or from [upstream](https://github.com/NorskRegnesentral/text-anonymization-benchmark) into
`dataset/tab_echr/` — `dataset/README.md` has the commands and the md5s to check against.

In [ ]:
tab = get_dataset("tab_echr")          # == TabECHR(split="test")
print(tab)
for k, v in tab.stats().items():
    print(f"  {k:20s} {round(v, 2) if isinstance(v, float) else v}")

In [ ]:
doc = tab.select(n=1, seed=0)[0]
print(f"doc_id : {doc.doc_id}   ({len(doc.text)} chars, {len(doc.spans)} spans)")
print(f"types  : {doc.entity_types()}\n")

from collections import Counter
for t, n in Counter(s.entity_type for s in doc.spans).most_common():
    sample = [s.text for s in doc.spans_of([t])][:4]
    print(f"  {t:9s} {n:3d}  {sample}")

# spans are char offsets into `text` -- the same form fusit.trace speaks
print(f"\noffsets(ATTACK_ENTITY_TYPES)[:5] = {doc.offsets(ATTACK_ENTITY_TYPES)[:5]}")

In [ ]:
# what a redacted document looks like: bracket every private span
import textwrap

def bracket(text, spans, limit=700):
    out, prev = [], 0
    for s in sorted(spans, key=lambda s: s.start):
        out.append(text[prev:s.start]); out.append(f"[{s.entity_type}]"); prev = s.end
    out.append(text[prev:])
    joined = "".join(out)
    return textwrap.fill(joined[:limit], 100) + ("..." if len(joined) > limit else "")

print(bracket(doc.text, doc.spans))

### Cross-check against the raw json

Same treatment as section 6: re-derive the filter straight from the file and confirm the
loader agrees, then check every span's offsets actually point at the text it claims.

In [ ]:
import json

raw = json.load(open(tab.path))
print(f"raw records : {len(raw)}")
print(f"loaded docs : {len(tab.items())}")
assert len(raw) == len(tab.items())

expected = 0
for r in raw:
    mentions = r["annotations"][next(iter(r["annotations"]))]["entity_mentions"]
    expected += sum(1 for m in mentions if m["identifier_type"] in {"DIRECT", "QUASI"})
got = sum(len(d.spans) for d in tab.items())
print(f"spans, re-derived from the file : {expected}")
print(f"spans, from the loader          : {got}")
assert got == expected, "loader and re-derived filter disagree"

# every span must be a real slice of its document
bad = [(d.doc_id, s) for d in tab.items() for s in d.spans if d.text[s.start:s.end] != s.text]
print(f"\nspans whose offsets do not match their text: {len(bad)}")
assert not bad

dropped = sum(
    1 for r in raw
    for m in r["annotations"][next(iter(r["annotations"]))]["entity_mentions"]
    if m["identifier_type"] not in {"DIRECT", "QUASI"}
)
print(f"mentions dropped as NO_MASK: {dropped}")

### The two knobs, measured

`require_types` exists because the paper scopes its *attack* to PERSON/CODE/DATETIME only,
"as they appear consistently across all documents" — worth confirming rather than assuming.

In [ ]:
for mode in ("first", "union"):
    st = TabECHR("test", annotator=mode).stats()
    print(f"annotator={mode:6s} {st['num_entities']:6d} entities, {st['private_pct']:5.2f}% of chars private")

print()
full = tab.select()
scoped = tab.select(require_types=ATTACK_ENTITY_TYPES)
print(f"documents carrying all of {ATTACK_ENTITY_TYPES}: {len(scoped)}/{len(full)}")

print()
for split in TabECHR.SPLITS:
    try:
        st = TabECHR(split).stats()
        print(f"  {split:6s} {st['num_items']:5d} docs  {st['num_entities']:6d} entities  {st['private_pct']:5.2f}% private")
    except FileNotFoundError:
        print(f"  {split:6s} not fetched")

### This is not the paper's exact subset

Worth stating plainly, since the numbers are close enough to be mistaken for a reproduction.
Paper Table 2 reports 100 documents / 423,573 chars / 69,451 private (16.40%) / 4,773
entities. The full test split is what the cell below compares against — the paper used a
100-document subset it does not identify, so these will not line up, and any figure derived
from this loader is *our* corpus, not theirs.

In [ ]:
paper = {"docs": 100, "chars": 423_573, "private_chars": 69_451, "private_pct": 16.40, "entities": 4_773}
st = tab.stats()
ours = {"docs": st["num_items"], "chars": st["total_chars"], "private_chars": st["private_chars"],
        "private_pct": round(st["private_pct"], 2), "entities": st["num_entities"]}

print(f"{'':15s} {'paper (Table 2)':>16s} {'ours (test split)':>18s}")
print("-" * 52)
for k in paper:
    print(f"{k:15s} {paper[k]:>16,} {ours[k]:>18,}")
print("\nPrivate character counts are close (69,451 vs "
      f"{ours['private_chars']:,}) while totals are not, so their subset is not simply a "
      "prefix of this split.")

## 8. Failure modes

Bad input should say what to do about it, not raise a bare `FileNotFoundError` or `KeyError`.

In [ ]:
try:
    SynthPAI(path=Path("/tmp/definitely-not-here/synthpai.jsonl")).items()
except FileNotFoundError as e:
    print("missing file ->", e)

try:
    get_dataset("synthpal")          # typo
except ValueError as e:
    print("bad name     ->", e)

# Dataset is abstract: it declares `name`/`filename` but does not define them, so it cannot
# be instantiated at all. The error is an AttributeError from `DATASET_DIR / self.filename`
# rather than a NotImplementedError from load() -- blunt, but it fails immediately and only
# ever reaches someone subclassing wrongly.
try:
    Dataset()
except AttributeError as e:
    print("base class   -> not instantiable:", e)

## 9. Scratch

Space for whatever you are actually prototyping. `synthpai` / `synthetic` are loaded above.

In [ ]:
items = load_dataset("synthpai", n=3, seed=0)
for it in items:
    print(f"{it.username:22s} {len(it.text):6d} chars  {sorted(it.relevant_pii)}")